# Pertemuan 8: UTS Mini Project ML (End-to-End)

In [4]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
# 1 Problem & Dataset
# ==========================================================
print("=" * 60)
print("1) PROBLEM & DATASET")
print("=" * 60)
print("Masalah : Prediksi harga rumah di California")
print("Tujuan  : Mengembangkan model untuk memprediksi nilai rata-rata rumah")
print("Sumber  : sklearn.datasets.fetch_california_housing")

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["MedHouseVal"] = data.target  # Target dalam satuan $100,000

print(f"\nUkuran data  : {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Fitur        : {len(data.feature_names)} fitur numerik")
print(f"Target       : MedHouseVal (Median House Value)")
print(f"Statistik Target:")
print(df["MedHouseVal"].describe().round(2))

1) PROBLEM & DATASET
Masalah : Prediksi harga rumah di California
Tujuan  : Mengembangkan model untuk memprediksi nilai rata-rata rumah
Sumber  : sklearn.datasets.fetch_california_housing

Ukuran data  : 20640 baris, 9 kolom
Fitur        : 8 fitur numerik
Target       : MedHouseVal (Median House Value)
Statistik Target:
count    20640.00
mean         2.07
std          1.15
min          0.15
25%          1.20
50%          1.80
75%          2.65
max          5.00
Name: MedHouseVal, dtype: float64


In [6]:
# 2 EDA
# ==========================================================
print("\n" + "=" * 60)
print("2) EDA")
print("=" * 60)

# Cek missing value
print("\nMissing values:", df.isnull().sum().sum())

# Statistik deskriptif (5 fitur utama)
print("\nStatistik deskriptif (5 fitur pertama):")
print(df.iloc[:, :5].describe().round(2))

# 5 Visualisasi
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Viz 1: Distribusi Target
sns.histplot(df["MedHouseVal"], bins=30, kde=True, ax=axes[0, 0], color="skyblue")
axes[0, 0].set_title("1. Distribusi Median House Value")

# Viz 2: MedInc vs Target (Korelasi tertinggi)
axes[0, 1].scatter(df["MedInc"], df["MedHouseVal"], alpha=0.3, color="green", s=1)
axes[0, 1].set_title("2. Median Income vs Price")
axes[0, 1].set_xlabel("Median Income")
axes[0, 1].set_ylabel("Price")

# Viz 3: HouseAge Distribution
axes[0, 2].hist(df["HouseAge"], bins=20, color="orange", edgecolor="black")
axes[0, 2].set_title("3. Distribusi Umur Rumah")

# Viz 4: Boxplot MedHouseVal (Outlier detection)
sns.boxplot(x=df["MedHouseVal"], ax=axes[1, 0], color="salmon")
axes[1, 0].set_title("4. Boxplot Harga Rumah")

# Viz 5: Correlation Heatmap
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1, 1], annot_kws={"size": 8})
axes[1, 1].set_title("5. Heatmap Korelasi")

axes[1, 2].axis("off")

plt.suptitle("Exploratory Data Analysis - California Housing", fontsize=16)
plt.tight_layout()
plt.savefig("pertemuan08_eda_regression.png")
plt.close()
print("Visualisasi EDA disimpan ke pertemuan08_eda_regression.png")

# 5 Insight
print("\n5 Insight:")
print(f"  1. Fitur 'MedInc' (Median Income) memiliki korelasi positif terkuat ({df.corr()['MedInc']['MedHouseVal']:.2f}) terhadap harga.")
print(f"  2. Terdapat penumpukan data (cap) pada harga 5.0, menunjukkan nilai tersebut adalah batas maksimum dalam dataset.")
print(f"  3. Rata-rata umur rumah di dataset ini adalah {df['HouseAge'].mean():.1f} tahun.")
print(f"  4. Tidak ditemukan missing value, sehingga tidak diperlukan imputasi.")
print(f"  5. Fitur Latitude dan Longitude menunjukkan sebaran geografis yang luas di California.")


2) EDA

Missing values: 0

Statistik deskriptif (5 fitur pertama):
         MedInc  HouseAge  AveRooms  AveBedrms  Population
count  20640.00  20640.00  20640.00   20640.00    20640.00
mean       3.87     28.64      5.43       1.10     1425.48
std        1.90     12.59      2.47       0.47     1132.46
min        0.50      1.00      0.85       0.33        3.00
25%        2.56     18.00      4.44       1.01      787.00
50%        3.53     29.00      5.23       1.05     1166.00
75%        4.74     37.00      6.05       1.10     1725.00
max       15.00     52.00    141.91      34.07    35682.00
Visualisasi EDA disimpan ke pertemuan08_eda_regression.png

5 Insight:
  1. Fitur 'MedInc' (Median Income) memiliki korelasi positif terkuat (0.69) terhadap harga.
  2. Terdapat penumpukan data (cap) pada harga 5.0, menunjukkan nilai tersebut adalah batas maksimum dalam dataset.
  3. Rata-rata umur rumah di dataset ini adalah 28.6 tahun.
  4. Tidak ditemukan missing value, sehingga tidak diperlukan

In [7]:
# 3 PREPROCESSING
# ==========================================================
print("\n" + "=" * 60)
print("3) PREPROCESSING")
print("=" * 60)

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# Split data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} sampel | Test: {X_test.shape[0]} sampel")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler diterapkan untuk menyeimbangkan skala fitur numerik.")


3) PREPROCESSING
Train: 16512 sampel | Test: 4128 sampel
StandardScaler diterapkan untuk menyeimbangkan skala fitur numerik.


In [8]:
# 4 MODELING — minimal 3 model
# ==========================================================
print("\n" + "=" * 60)
print("4) MODELING")
print("=" * 60)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    trained_models[name] = model

# Tabel perbandingan
results_df = pd.DataFrame(results)
print("\nTabel Perbandingan Performa:")
print(results_df.round(4).to_string(index=False))

# Detail model terbaik (berdasarkan R2 tertinggi)
best_name = results_df.loc[results_df['R2'].idxmax(), 'Model']
best_model = trained_models[best_name]
print(f"\nModel terbaik dipilih: {best_name}")


4) MODELING

Tabel Perbandingan Performa:
                  Model    MAE   RMSE     R2
      Linear Regression 0.5332 0.7456 0.5758
Decision Tree Regressor 0.4330 0.6442 0.6833
Random Forest Regressor 0.3675 0.5449 0.7734

Model terbaik dipilih: Random Forest Regressor


In [9]:
# 5 TUNING (GridSearchCV pada model terbaik)
# ==========================================================
print("\n" + "=" * 60)
print("5) TUNING")
print("=" * 60)

print(f"Melakukan tuning pada {best_name}...")
if "Random Forest" in best_name:
    param_grid = {
        "n_estimators": [50, 100],
        "max_depth": [10, 20],
        "min_samples_split": [2, 5]
    }
    grid = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring="r2", n_jobs=-1)
elif "Decision Tree" in best_name:
    param_grid = {
        "max_depth": [5, 10, 20, None],
        "min_samples_leaf": [1, 2, 4]
    }
    grid = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=3, scoring="r2", n_jobs=-1)
else:
    print("Linear Regression tidak memerlukan tuning hyperparameter intensif dalam skenario ini.")
    grid = None

if grid:
    grid.fit(X_train_scaled, y_train)
    print(f"Best params: {grid.best_params_}")
    tuned_pred = grid.predict(X_test_scaled)
    tuned_r2 = r2_score(y_test, tuned_pred)
    print(f"R2 Score setelah tuning: {tuned_r2:.4f}")
else:
    tuned_r2 = results_df.loc[results_df['Model'] == best_name, 'R2'].values[0]


5) TUNING
Melakukan tuning pada Random Forest Regressor...
Best params: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 100}
R2 Score setelah tuning: 0.8049


In [10]:
# 6 KESIMPULAN
# ==========================================================
print("\n" + "=" * 60)
print("6) KESIMPULAN")
print("=" * 60)
print(f"• Model terbaik: {best_name}")
print(f"• Skor R2 Akhir: {tuned_r2:.4f}")
print("• Insight Bisnis: Median Income adalah prediktor harga rumah yang paling signifikan.")
print("• Rekomendasi: Gunakan model Random Forest untuk akurasi prediksi harga yang lebih stabil.")
print("• Tantangan: Adanya 'price capping' pada data target sedikit memengaruhi linearitas model.")

print("\n✅ Pertemuan 08 (UTS Mini Project - Regression) selesai.")


6) KESIMPULAN
• Model terbaik: Random Forest Regressor
• Skor R2 Akhir: 0.8049
• Insight Bisnis: Median Income adalah prediktor harga rumah yang paling signifikan.
• Rekomendasi: Gunakan model Random Forest untuk akurasi prediksi harga yang lebih stabil.
• Tantangan: Adanya 'price capping' pada data target sedikit memengaruhi linearitas model.

✅ Pertemuan 08 (UTS Mini Project - Regression) selesai.
